# 2 — Training notebook (`train.ipynb`)

One notebook per direction, in `notebooks/architectures/<direction>/`. Every configuration the
direction tries is a variant inside this notebook, not a separate notebook.

**Rules** (from `lie-detector/AGENTS.md`)

1. Don't touch `data/processed/splits/` or `lib/` — if you need a change there, tell the group.
2. The model lives in `model.py` beside this notebook.
3. **Results come from the five folds.** This notebook never touches the test set.
4. Save predictions per clip: they are what makes the directions comparable.

**State.** Implemented: `late` (late fusion), the baseline of
[docs/trainable-module.md](docs/trainable-module.md). The next variants are classes in `model.py`
plus one line in `VARIANT`; the loop below does not change.

## 0 — Setup

In [68]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [82]:
# The repo is needed for two things that are NOT on Drive: the `lib/` code and the
# frozen splits (they are versioned in git, that is what makes them frozen).
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')
#%load_ext autoreload
#%autoreload 2

/content/DeepLearningCourse/lie-detector


## 1 — Settings

In [83]:
from pathlib import Path

# The one shared folder. Same path for everybody: add a Drive shortcut named
# "dolos" in your My Drive and this resolves without touching anything else.
DATA = Path("/content/drive/MyDrive/dolos")
print(DATA, "->", sorted(p.name for p in DATA.glob("*")))

/content/drive/MyDrive/dolos -> ['audio', 'cached_features', 'clips', 'dataset.csv', 'dolos_annotations.zip', 'splits', 'splits_v2']


In [84]:
from lib.dataset import cache_dir

DIRECTION = "tia"                  # the row of the results board
VIDEO_ENCODER = "dinov2-small"          # (extract_features.ipynb): 16 tokens, 384-d CLS
#VIDEO_ENCODER = "hsemotion-b0-vgaf"    # (extract_features.ipynb): 16 tokens, 1280-d, supervised
#VIDEO_ENCODER = "hsemotion-b0-vgaf-emotions"   # the same encoder: 16 tokens, 8-d posteriors
#VIDEO_ENCODER = "vjepa2-vitl"          # (extract_vjepa2.ipynb): 32 tokens, 1024-d
AUDIO_ENCODER = "wavlm-base-plus"
VIDEO_CACHE = cache_dir(VIDEO_ENCODER, DATA)
AUDIO_CACHE = cache_dir(AUDIO_ENCODER, DATA)

VARIANT = "late"                   # a key of model.VARIANTS
SEED = 0
BATCH_SIZE = 16
EPOCHS = 30
PATIENCE = 8
FOLDS = 5
SEEDS = [0, 1, 2]                        # [0, 1, 2] before the board row: one seed is a smoke test
SPLIT = "subject"                  # "subject" (our protocol) or "official" (the paper's 3-fold)
PROTOCOL = "v1"                     # "v1" (board) or "v2" (needs SPLIT="subject")
SPLITS = DATA / ("splits_v2" if PROTOCOL == "v2" else "splits")
CAPTAINS = "include"               # "include" as shipped, or "exclude" for the ablation
READ_WORKERS = 64                  # parallel readers for the Drive mount: it is latency-bound

# The model is tiny, so CPU works; the GPU is what keeps the loop short when the
# variants grow (and it is what the board row reports). MPS is for a local Mac.
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print(VIDEO_CACHE)
print(AUDIO_CACHE)

# `model.py` and `export.py` live beside this notebook, which is not the folder the setup
# cell `%cd`s into: put it on the path once, here, and give it a name the rest can use.
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "model.py").exists():
    NOTEBOOK_DIR = Path.cwd() / "notebooks/architectures/cross-modal-fusion"
sys.path.insert(0, str(NOTEBOOK_DIR))
print("notebook dir:", NOTEBOOK_DIR)

device: cuda Tesla T4
/content/drive/MyDrive/dolos/cached_features/dinov2-small
/content/drive/MyDrive/dolos/cached_features/wavlm-base-plus


## 2 — The frozen splits

`Protocol.load(SPLITS)` reads the protocol from the shared folder and verifies the manifest hash:
a file edited by hand raises instead of silently giving you a different split.

`PROTOCOL = "v1"` is the board protocol. `"v2"` is the purity-balanced ablation described in
[`docs/splits-v2.md`](../../../docs/splits-v2.md): same test set, folds re-cut so no fold is made of pure
subjects. The official 3-fold exists only in v1. Do not mix v1 and v2 rows on the board.


In [85]:
from lib.dataset import Protocol, Clips

if SPLIT == "official" and PROTOCOL != "v1":
    raise SystemExit("the official 3-fold exists only in v1: set PROTOCOL = 'v1'")
print(f"protocol: {PROTOCOL} -> {SPLITS}")
folds = Protocol.load(SPLITS)
clips = Clips.load(DATA)
print(clips.describe_source())   # post it: same fingerprint = same dataset

N_FOLDS = 3 if SPLIT == "official" else folds.folds
if SPLIT == "official":
    print("⚠️  official 3-fold: clip-level — 147 subjects and every episode sit on both sides.")
    print("    Reproduction of the paper's protocol: reported separately, never on the board.")


def fold_parts(fold):
    """(train, val, holdout) for the current protocol. The official split has no val.

    With CAPTAINS = "exclude" the two subjects that every fold pins to training are dropped
    from it: they are 468 clips, about half of a fold's training set, and they are never
    evaluated either way.
    """
    if SPLIT == "official":
        train_ids, test_ids = folds.official(fold)
        return train_ids, None, test_ids
    train_ids, val_ids, hold_ids = folds.subject(fold)
    if CAPTAINS == "exclude":
        pinned = set(folds._splits[f"subject_fold{fold}.json"]["always_train"]["clips"])
        train_ids = [clip_id for clip_id in train_ids if clip_id not in pinned]
    return train_ids, val_ids, hold_ids


train, val, holdout = fold_parts(1)
split_ids = [c for fold in range(1, FOLDS + 1)
             for part in folds.subject(fold) for c in part] + folds.test()
missing = clips.missing_from(split_ids)
print(f"{len(missing)} split clips are absent from dataset.csv")
if missing:
    print("  first:", missing[:3], "-> the splits and the data do not match, stop here")

for name, ids in (("train", train), ("val", val), ("holdout", holdout)):
    print(f"{name:8}", "(none)" if ids is None else clips.describe(ids))

protocol: v1 -> /content/drive/MyDrive/dolos/splits
dataset.csv sha256:8fa198b1fe71  1678 clips  1452 with media
0 split clips are absent from dataset.csv
train    {'clips': 1011, 'subjects': 94, 'deception': 56.1, 'pos_weight': 0.783, 'female': 17.0, 'majority_baseline': 56.1}
val      {'clips': 109, 'subjects': 16, 'deception': 47.7, 'pos_weight': 1.096, 'female': 28.4, 'majority_baseline': 52.3}
holdout  {'clips': 177, 'subjects': 27, 'deception': 48.0, 'pos_weight': 1.082, 'female': 28.2, 'majority_baseline': 52.0}


## 3 — The feature caches

TIA needs **both** arrays for a clip, so the loader is paired: `(video, audio) -> label`.
The splits are frozen on `dataset.csv` while the caches only cover the clips whose media and
audio are on disk, so the two are intersected once, up front.

The whole working set is read into RAM **once**. This matters: `CachedFeatures` stats every
clip it is handed — a network round trip each on the Drive mount — and reads whatever split it
is given, so building one per split (and three per fold inside `train_one_fold`) re-reads a
330 MB cache dozens of times. One pass here, then every `loader_for` is a dict lookup.

If a cache is missing the notebook falls back to deterministic pseudo-features with the real
shapes, so the loop can be tested end to end. **The numbers are meaningless in that mode.**

In [86]:
import hashlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
from torch.utils.data import Dataset
from lib.dataset import make_loader

DEBUG = not (VIDEO_CACHE.exists() and AUDIO_CACHE.exists())
if DEBUG:
    absent = [name for name, path in (("video", VIDEO_CACHE),
                                      ("audio", AUDIO_CACHE)) if not path.exists()]
    print("⚠️  no cache for:", ", ".join(absent), "-> pseudo-features, numbers are meaningless")


def working_ids():
    """Every clip the current protocol touches, once each, in a stable order.

    In official mode this is the union of the paper's train/test pairs, which covers the
    whole dataset — the subject folds leave the virgin test set out, and those clips are
    part of the official test folds.
    """
    ids = []
    for fold in range(1, N_FOLDS + 1):
        parts = folds.official(fold) if SPLIT == "official" else folds.subject(fold)
        for part in parts:
            ids += part
    return list(dict.fromkeys(ids))


class Features:
    """Both caches in RAM, addressed by clip id, read once — in parallel.

    `CachedFeatures` is the general loader, but it stats every clip it is handed and reads
    whatever split it is given: one construction per split, three per fold, re-reads the
    cache dozens of times. Here every array is read once, from a pool of threads: the Drive
    mount is latency-bound, so 16 readers overlap the round trips instead of queueing them.
    A file that is not there is counted, not fatal.
    """

    def __init__(self, ids, video_cache, audio_cache, workers=READ_WORKERS):
        self.video, missing_video = self._read_all(video_cache, ids, "video", workers)
        self.audio, missing_audio = self._read_all(audio_cache, ids, "audio", workers)
        self.clip_ids = [c for c in ids if c in self.video and c in self.audio]
        self.index = {c: i for i, c in enumerate(self.clip_ids)}
        print(f"video {len(self.video)} read, {missing_video} missing | "
              f"audio {len(self.audio)} read, {missing_audio} missing")
        print(f"usable: {len(self.clip_ids)}/{len(ids)} clips")

    @staticmethod
    def _read(cache, clip_id):
        return torch.from_numpy(np.load(cache / f"{clip_id}.npy")).float()

    def _read_all(self, cache, ids, label, workers):
        def load(clip_id):
            try:
                return clip_id, self._read(cache, clip_id)
            except OSError:
                return clip_id, None

        out, missing, step = {}, 0, max(1, len(ids) // 8)
        with ThreadPoolExecutor(workers) as pool:
            for done, (clip_id, tensor) in enumerate(pool.map(load, ids), 1):
                if tensor is None:
                    missing += 1
                else:
                    out[clip_id] = tensor
                if done % step == 0:
                    print(f"  {label}: {done}/{len(ids)}", flush=True)
        return out, missing

    def subset(self, ids):
        return [c for c in ids if c in self.index]


class Paired(Dataset):
    """`(video, audio) -> label` for clips the store already holds."""

    def __init__(self, features, ids):
        self.features = features
        self.clip_ids = list(ids)

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, index):
        clip_id = self.clip_ids[index]
        return ((self.features.video[clip_id], self.features.audio[clip_id]),
                torch.tensor(clips[clip_id].y, dtype=torch.long))


class PseudoPaired(Dataset):
    """Deterministic stand-in with the real shapes: proves the loop, not the model."""

    def __init__(self, ids, clips, frames=16, video_dim=384, tokens=64, audio_dim=768):
        self.clip_ids = clips.keep(ids)
        self.clips = clips
        self.shapes = ((frames, video_dim), (tokens, audio_dim))

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, index):
        clip_id = self.clip_ids[index]
        rng = np.random.default_rng(int(hashlib.md5(clip_id.encode()).hexdigest()[:8], 16))
        video, audio = (torch.tensor(rng.normal(size=shape), dtype=torch.float32)
                        for shape in self.shapes)
        return (video, audio), torch.tensor(self.clips[clip_id].y, dtype=torch.long)


if DEBUG:
    FEATURES = None
else:
    wanted = clips.keep(working_ids())
    print(f"{len(wanted)} working clips, reading both caches once")
    FEATURES = Features(wanted, VIDEO_CACHE, AUDIO_CACHE)
    megabytes = sum(FEATURES.video[c].numel() + FEATURES.audio[c].numel()
                    for c in FEATURES.clip_ids) * 4 / 1e6
    print(f"in RAM: {len(FEATURES.clip_ids)} clips, {megabytes:.0f} MB")


def loader_for(ids, shuffle, seed=SEED):
    dataset = (PseudoPaired(ids, clips) if DEBUG
               else Paired(FEATURES, FEATURES.subset(ids)))
    return make_loader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, seed=seed)

1297 working clips, reading the three caches once
  video: 162/1297
  video: 324/1297
  video: 486/1297
  video: 648/1297
  video: 810/1297
  video: 972/1297
  video: 1134/1297
  video: 1296/1297
  audio: 162/1297
  audio: 324/1297
  audio: 486/1297
  audio: 648/1297
  audio: 810/1297
  audio: 972/1297
  audio: 1134/1297
  audio: 1296/1297
  text: 162/1297
  text: 324/1297
  text: 486/1297
  text: 648/1297
  text: 810/1297
  text: 972/1297
  text: 1134/1297
  text: 1296/1297
video 1158 read, 139 missing | audio 1158 read, 139 missing | text 1158 read, 139 missing
usable: 1158/1297 clips
in RAM: 1158 clips, 258 MB


In [87]:
train_ids, val_ids, holdout_ids = fold_parts(1)
for name, ids in (("train", train_ids), ("val", val_ids), ("holdout", holdout_ids)):
    if ids is None:
        print(f"{name:8} (the official split has no validation set)")
        continue
    print(f"{name:8} {len(ids):4d} split clips -> {len(loader_for(ids, False).dataset):4d} usable")

(video, audio), y = next(iter(loader_for(train_ids, shuffle=False)))
print("batch", tuple(video.shape), tuple(audio.shape), tuple(y.shape))

train    1011 split clips ->  904 usable
val       109 split clips ->   92 usable
holdout   177 split clips ->  162 usable
batch (16, 16, 384) (16, 64, 768) (16, 384) (16,)


### How the folds differ

The same fold can look easier or harder for reasons that have nothing to do with the model: a
holdout with more deception, or a validation set that happens to resemble it. This prints the
composition of every fold, so a fold that wins everywhere can be explained before it is interpreted.

In [88]:
import pandas as pd

rows = []
for fold in range(1, N_FOLDS + 1):
    parts = fold_parts(fold)
    row = {"fold": fold}
    for name, ids in zip(("train", "val", "holdout"), parts):
        ids = clips.keep(ids) if ids is not None else []
        ids = FEATURES.subset(ids) if FEATURES is not None else ids
        stats = clips.describe(ids)
        row[f"{name}_n"] = stats["clips"]
        row[f"{name}_dec"] = stats["deception"]
        row[f"{name}_maj"] = stats["majority_baseline"]
    rows.append(row)

folds_frame = pd.DataFrame(rows)
print(folds_frame.round(1).to_string(index=False))

 fold  train_n  train_dec  train_maj  val_n  val_dec  val_maj  holdout_n  holdout_dec  holdout_maj
    1      904       54.9       54.9     92     46.7     53.3        162         50.0         50.0
    2      906       55.4       55.4     95     50.5     50.5        157         44.6         55.4
    3      895       56.0       56.0     99     44.4     55.6        164         45.7         54.3
    4      892       54.3       54.3    101     50.5     50.5        165         51.5         51.5
    5      907       55.5       55.5     99     49.5     50.5        152         44.7         55.3


## 4 — The model

`model.py` holds the variants and the registry; this cell only picks one. The interface is
`forward(video, audio) -> (B,)`, one logit per clip, which is what the loop below expects.

In [89]:
import importlib
import sys
from pathlib import Path

import model

importlib.reload(model)          # pick up edits to model.py without restarting the kernel
from model import VARIANTS

if VARIANT not in VARIANTS:
    raise KeyError(
        f"{VARIANT!r} is not in {NOTEBOOK_DIR / 'model.py'}: available {sorted(VARIANTS)}. "
        "Re-upload model.py, or fix VARIANT in the settings cell."
    )

# `model.py` builds its first layer from module constants; point them at the widths of the
# caches actually loaded. DINOv2 (384) + WavLM (768) is the default; a V-JEPA 2 video
# cache makes the first number 1024 with no other change.
if FEATURES is not None:
    sample = FEATURES.clip_ids[0]
    model.configure(video_dim=FEATURES.video[sample].shape[-1],
                    audio_dim=FEATURES.audio[sample].shape[-1])
print(f"model widths: video {model.VIDEO_DIM} | audio {model.AUDIO_DIM}")

net = VARIANTS[VARIANT]().to(DEVICE)
params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(net)
print(f"{VARIANT}: {params:,} trainable parameters on {DEVICE}")

model widths: video 384 | audio 768 | text 384
LateFusion(
  (video): Linear(in_features=384, out_features=512, bias=True)
  (audio): Linear(in_features=768, out_features=512, bias=True)
  (head): Sequential(
    (0): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=1024, out_features=128, bias=True)
    (3): GELU(approximate='none')
    (4): Dropout(p=0.3, inplace=False)
    (5): Linear(in_features=128, out_features=1, bias=True)
  )
)
late: 724,225 trainable parameters on cuda


## 5 — One fold

Early stopping on validation AUC, `pos_weight` to compensate the class prior (the training
folds are ~56% deception, the holdouts ~47%), and the decision threshold chosen on `val` — never
on the holdout.

In [90]:
import copy
import time

import torch
from lib.metrics import best_threshold, evaluate


def train_one_fold(fold, seed=SEED, verbose=True):
    train_ids, val_ids, hold_ids = fold_parts(fold)
    dl_train = loader_for(train_ids, True, seed=seed)
    dl_val = loader_for(val_ids, False) if val_ids is not None else None
    dl_hold = loader_for(hold_ids, False)

    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    net = VARIANTS[VARIANT]().to(DEVICE)
    optimiser = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=5e-2)
    # pos_weight is a buffer of the loss, so `.to(DEVICE)` moves it with the module
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=clips.class_weights(train_ids)).to(DEVICE)

    best_score, best_state, best_thr, waited = -1.0, None, 0.5, 0
    started = time.perf_counter()
    epoch = -1
    for epoch in range(EPOCHS):
        net.train()
        for (video, audio), y in dl_train:
            video, audio, y = video.to(DEVICE), audio.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad()
            loss = loss_fn(net(video, audio), y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimiser.step()

        if dl_val is None:                    # official split: no early stopping, no val threshold
            continue
        net.eval()
        probs, targets = [], []
        with torch.no_grad():
            for (video, audio), y in dl_val:
                probs += torch.sigmoid(net(video.to(DEVICE), audio.to(DEVICE))).tolist()
                targets += y.tolist()
        score = evaluate(targets, probs)["auc"]
        score = score if score == score else 0.0          # NaN if one class only

        if score > best_score:
            best_score, best_state, waited = score, copy.deepcopy(net.state_dict()), 0
            best_thr = best_threshold(targets, probs)
        else:
            waited += 1
            if waited >= PATIENCE:
                break
    seconds = time.perf_counter() - started

    if best_state is not None:
        net.load_state_dict(best_state)
    else:
        best_thr = 0.5                        # the paper's operating point: argmax of the logits
    net.eval()
    probs, targets = [], []
    with torch.no_grad():
        for (video, audio), y in dl_hold:
            probs += torch.sigmoid(net(video.to(DEVICE), audio.to(DEVICE))).tolist()
            targets += y.tolist()

    # the dataset, not the split: clips without a cached array were dropped by `usable_ids`
    ids = dl_hold.dataset.clip_ids
    assert len(ids) == len(probs) == len(targets), (len(ids), len(probs), len(targets))

    metrics = evaluate(targets, probs, threshold=best_thr)
    metrics["seconds"] = seconds
    metrics["epochs"] = epoch + 1
    if verbose:
        print(f"fold {fold} seed {seed}: ACC {metrics['acc']*100:5.1f}  F1 {metrics['f1']*100:5.1f}  "
              f"AUC {metrics['auc']*100:5.1f}  thr {best_thr:.2f}  "
              f"{seconds:4.0f}s  ({epoch + 1} epochs)")
    return metrics, ids, targets, probs

## 6 — All folds

The number that goes on the board is the **mean over the five folds**; the spread tells you how
much of it is noise. One seed is a smoke test — the board row wants `SEEDS = [0, 1, 2]`.

In [91]:
import numpy as np
import pandas as pd

# what is about to run — a stale VARIANT in the kernel is otherwise invisible until the numbers
# come out identical to the previous variant
print(f"{VARIANT} ({VARIANTS[VARIANT].__name__}, {params:,} params) on {SPLIT}/{PROTOCOL}, "
      f"captains {CAPTAINS}: {len(SEEDS)} seed(s) x {N_FOLDS} folds")

results, pooled = [], []
for seed in SEEDS:
    for fold in range(1, N_FOLDS + 1):
        metrics, ids, targets, probs = train_one_fold(fold, seed=seed)
        results.append({"seed": seed, "fold": fold, **metrics})
        pooled.append(pd.DataFrame({"clip_id": ids, "y_true": targets,
                                    "y_prob": probs, "seed": seed, "fold": fold}))

summary = pd.DataFrame(results)
print(summary[["seed", "fold", "acc", "f1", "auc", "balanced_acc", "seconds"]].round(3)
      .to_string(index=False))
print(f"\nCV  ACC {summary.acc.mean()*100:.1f} ± {summary.acc.std()*100:.1f}  "
      f"F1 {summary.f1.mean()*100:.1f}  AUC {summary.auc.mean()*100:.1f}  "
      f"balanced {summary.balanced_acc.mean()*100:.1f}")

late (LateFusion, 724,225 params) on subject/v1, captains include: 3 seed(s) x 5 folds
fold 1 seed 0: ACC  56.8  F1  53.3  AUC  63.4  thr 0.72    19s  (19 epochs)
fold 2 seed 0: ACC  50.3  F1  57.1  AUC  57.5  thr 0.43    13s  (15 epochs)
fold 3 seed 0: ACC  51.8  F1  54.9  AUC  55.9  thr 0.29    12s  (15 epochs)
fold 4 seed 0: ACC  54.5  F1  61.9  AUC  48.9  thr 0.10    20s  (24 epochs)
fold 5 seed 0: ACC  52.0  F1  59.2  AUC  56.2  thr 0.21    10s  (11 epochs)
fold 1 seed 1: ACC  63.6  F1  57.6  AUC  68.8  thr 0.78    21s  (29 epochs)
fold 2 seed 1: ACC  40.8  F1  43.6  AUC  45.5  thr 0.48    10s  (11 epochs)
fold 3 seed 1: ACC  49.4  F1  46.5  AUC  48.4  thr 0.35     6s  (9 epochs)
fold 4 seed 1: ACC  57.0  F1  64.7  AUC  49.6  thr 0.08    22s  (24 epochs)


: 

## 7 — Pooled predictions and per-gender breakdown

Every working-set clip is predicted exactly once: the fold holdouts partition the working set.
With several seeds the probabilities are averaged per clip, so the paired comparison with the
other directions (McNemar) still has one prediction per clip.

In [ ]:
from lib.metrics import evaluate, majority_baseline

preds = pd.concat(pooled, ignore_index=True)
per_clip = (preds.groupby("clip_id", as_index=False)
                 .agg(y_true=("y_true", "first"), y_prob=("y_prob", "mean")))

overall = evaluate(per_clip.y_true, per_clip.y_prob)   # at 0.5, for reference only
base = majority_baseline(per_clip.y_true)
print(f"pooled ACC {overall['acc']*100:.1f}  balanced {overall['balanced_acc']*100:.1f}  "
      f"| majority baseline {base*100:.1f}")
# AUC is the balance-free number; the gap between the mean probability and the true rate
# is how much of the training prior the model still carries onto the holdout
print(f"pooled AUC {overall['auc']*100:.1f}  | mean predicted prob {per_clip.y_prob.mean():.3f} "
      f"vs true rate {per_clip.y_true.mean():.3f}")

# one file per variant, so two configurations can be compared on the same clips;
# copy the winner to cv_<direction>.csv when it goes on the board
out = Path("preds"); out.mkdir(exist_ok=True)
suffix = "" if CAPTAINS == "include" else f"_{CAPTAINS}"
if PROTOCOL != "v1":
    suffix += f"_{PROTOCOL}"
target = out / f"cv_{DIRECTION}_{VARIANT}_{SPLIT}{suffix}.csv"
per_clip.to_csv(target, index=False)
print("saved", target)

pooled ACC 55.0  balanced 53.8  | majority baseline 52.6
pooled AUC 53.3  | mean predicted prob 0.420 vs true rate 0.474
saved preds/cv_tia_late_subject.csv


In [ ]:
meta = pd.read_csv(DATA / "dataset.csv", dtype=str).set_index("clip_id")
per_clip["gender"] = per_clip.clip_id.map(meta.gender)
for gender, group in per_clip.groupby("gender"):
    m = evaluate(group.y_true, group.y_prob)
    print(f"{gender:7} n={m['n']:4d}  ACC {m['acc']*100:5.1f}  balanced {m['balanced_acc']*100:5.1f}")

female  n= 224  ACC  53.1  balanced  52.0
male    n= 576  ACC  55.7  balanced  54.6


## 8 — Row for the results board

`lib/metrics.format_row` prints the row in the board's column order. Paste it into
`## Results board` of `lie-detector/README.md` with your name and one line of interpretation.

In [ ]:
from lib.metrics import format_row

board = {"acc": summary.acc.mean(), "acc_std": summary.acc.std(),
         "f1": summary.f1.mean(), "auc": summary.auc.mean(),
         "balanced_acc": summary.balanced_acc.mean()}
row = format_row(DIRECTION, board, owner="mattsays", method=VARIANT, params=params,
                 majority=base, runtime=f"{summary.seconds.mean():.0f} s/fold ({DEVICE})")

if PROTOCOL != "v1":
    print("v2 is an ablation: this row is not for the results board.")
elif SPLIT == "official":
    print("official 3-fold: clip-level, not comparable to the board. Report it separately.")
else:
    print(row)

| tia | mattsays | late | 312,897 | 50.9 ± 5.8 | 45.6 | 51.1 | 50.8 | 52.6 | 11 s/fold (cuda) |


## 9 — Save the run

`results/` is what the report reads and what the other directions compare against, so a run that
is not written there does not exist: the numbers in the README would quote a Colab kernel that
nobody can inspect. The layout is owned by [`results/README.md`](results/README.md); this cell
only supplies the three objects it asks for — the per-fold table, the pooled predictions, the
settings, and the two fingerprints (`dataset.csv` sha256, split manifest sha256).

Two guards. A run on pseudo-features is refused, because its numbers are meaningless. A run
whose git tree is dirty records `git_dirty: true` in `run.json` rather than pretending the
commit is the code that produced it — say what changed.


In [ ]:
from export import describe, run_config, save_run

config = run_config(
    direction=DIRECTION,
    variant=VARIANT,
    split=SPLIT,
    protocol=PROTOCOL,
    captains=CAPTAINS,
    owner="mattsays",
    folds=N_FOLDS,
    seeds=list(SEEDS),
    encoders={"video": VIDEO_ENCODER, "audio": AUDIO_ENCODER},
    widths={"video": model.VIDEO_DIM, "audio": model.AUDIO_DIM},
    trainable_params=int(params),
    training={"batch_size": BATCH_SIZE, "epochs": EPOCHS, "patience": PATIENCE,
              "optimiser": "AdamW", "lr": 1e-3, "weight_decay": 5e-2,
              "loss": "BCEWithLogitsLoss(pos_weight=training-fold prevalence)"},
    device=str(DEVICE),
)
config.update(describe(clips, folds))

if DEBUG:
    print("⚠️  pseudo-features: refusing to write a run whose numbers mean nothing")
else:
    save_run(summary, per_clip, config, root=NOTEBOOK_DIR / "results")
    if config["git_dirty"]:
        print("⚠️  uncommitted changes: run.json records git_dirty = true")
